# Import Libraries and Data

In [ ]:
!pip install optuna
!pip install category_encoders
!pip install catboost
!pip install shap
!pip install feature-engine
!pip install tsfresh

In [ ]:
#import polars as pl
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error, silhouette_score
from sklearn.model_selection import cross_val_score
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.model_selection import StratifiedKFold, GroupKFold, KFold
from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.linear_model import Lasso, LassoCV, Ridge, RidgeCV
from sklearn.neighbors import KNeighborsRegressor

#from feature_engine.encoding import RareLabelEncoder
#from feature_engine.outliers import Winsorizer
import tsfresh

import lightgbm
from lightgbm import early_stopping
from lightgbm import log_evaluation
import xgboost
from catboost import CatBoostRegressor, Pool, cv
import optuna

import re
from category_encoders import PolynomialEncoder, CountEncoder, TargetEncoder, GLMMEncoder, JamesSteinEncoder, MEstimateEncoder, LeaveOneOutEncoder, CatBoostEncoder, WOEEncoder, HelmertEncoder, SumEncoder
import shap

# Read-in Data

In [ ]:
# Mount drive so that we can access google drive files
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [ ]:
sample_sub = pd.read_csv('/drive/My Drive/Colab Notebooks/Zindi - Huawei/sample_submission.csv')
zindi_sub = pd.read_csv('/drive/My Drive/Colab Notebooks/Zindi - Huawei/SampleSubmission.csv')

cell_data = pd.read_csv('/drive/My Drive/Colab Notebooks/Zindi - Huawei/cell_level_data.csv')
station_data = pd.read_csv('/drive/My Drive/Colab Notebooks/Zindi - Huawei/station_configurations.csv')

train_labels = pd.read_csv('/drive/My Drive/Colab Notebooks/Zindi - Huawei/train_labels.csv')
train_labels = train_labels.merge(sample_sub[['BS','w']].drop_duplicates(), how='left', left_on=['BS'], right_on=['BS'])
train_labels['w'] = train_labels['w'].fillna(1)
train_labels['Time_BS'] = train_labels['Time'].astype(str) + "_" + train_labels['BS'].astype(str)

time_stations = cell_data[['Time','BS']].drop_duplicates()
time_stations['Time_BS'] = time_stations['Time'].astype(str) + "_" + time_stations['BS'].astype(str)

train_flags = train_labels['Time'].astype(str) + "_" + train_labels['BS'].astype(str)

# Pre-processing

In [ ]:
# Merge station_data with cell_data
cell_data_merged = cell_data.merge(station_data, how='left', left_on=['BS','CellName'], right_on=['BS','CellName'])
cell_data_merged = pd.concat([pd.Series(cell_data_merged['Time'].astype(str) + "_" + cell_data_merged['BS'].astype(str),name='Time_BS'), cell_data_merged],axis=1)


# Now iterate over it to transpose the data by cell number
cell_complete = time_stations.copy()

for cell in cell_data_merged['CellName'].unique():

    cell_df = cell_data_merged[cell_data_merged['CellName']==cell].drop(['Mode','RUType','CellName'],axis=1)

    cell_complete = cell_complete.merge(cell_df, how='left', left_on=['Time_BS'], right_on=['Time_BS'], suffixes=('', '_'+str(cell)))
    cell_complete = cell_complete.drop(['Time'+'_'+str(cell), 'BS'+'_'+str(cell)],axis=1)



# Mode, and RUType are unique to each station so we only need one column for each of these
temp_station = station_data[['BS','Mode','RUType']].drop_duplicates()
cell_complete = cell_complete.merge(temp_station, how='left', left_on=['BS'], right_on=['BS'])
#cell_complete['mode_rutype'] = cell_complete['Mode'] + '_' + cell_complete['RUType']


# Add flag to determine if data point is in training or testing data
cell_complete['train_flag'] = np.where(cell_complete['Time_BS'].isin(train_flags)==True, 1, 0)


# Add datetime features
cell_complete['Time'] = cell_complete['Time'].astype('datetime64[ns]')
cell_complete['day'] = cell_complete['Time'].dt.day
cell_complete['day_of_week'] = cell_complete['Time'].dt.day_of_week
cell_complete['hour'] = cell_complete['Time'].dt.hour

#cell_complete['hour_sin'] = np.sin(2 * np.pi * cell_complete['hour']/24.0)
#cell_complete['hour_cos'] = np.cos(2 * np.pi * cell_complete['hour']/24.0)


# Add cell counts for each station (seems to work best when just using the number of cells)
cell_counts = station_data.groupby('BS')['CellName'].nunique().reset_index()
cell_counts = dict(zip(cell_counts['BS'],cell_counts['CellName']))

cell_complete['num_cells'] = cell_complete['BS'].map(cell_counts)


# Group rare mode_types into a similar category (doesn't seem to affect things at all)
#cell_complete['mode_rutype'] = np.where((cell_complete['mode_rutype'].isin(['Mode1_Type11','Mode1_Type12'])==True) & (cell_complete['train_flag']==1), 'Mode1_Type10', cell_complete['mode_rutype'])
#test['mode_rutype'] = np.where(test['mode_rutype'].isin(['Mode1_Type11','Mode1_Type12'])==True,'Mode1_Type10',test['mode_rutype'])
cell_complete['RUType'] = np.where((cell_complete['RUType'].isin(['Type11','Type12'])==True) & (cell_complete['train_flag']==1), 'Type10', cell_complete['RUType'])
cell_complete['Frequency'] = np.where((cell_complete['Frequency']==979.998) & (cell_complete['train_flag']==1), 697.002, cell_complete['Frequency'])
cell_complete['Bandwidth'] = np.where((cell_complete['Bandwidth']==8) & (cell_complete['train_flag']==1), 5, cell_complete['Bandwidth'])
cell_complete['Antennas'] = np.where((cell_complete['Antennas']==64) & (cell_complete['train_flag']==1), 32, cell_complete['Antennas'])





# Add antenna total
ants = station_data.groupby('BS')['Antennas'].max().reset_index()
ants = dict(zip(ants['BS'],ants['Antennas']))

cell_complete['antenna_total'] = cell_complete['BS'].map(ants)
#cell_complete['antenna_total'] = cell_complete['antenna_total'].astype(str)
cell_complete.drop(['Antennas','Antennas_Cell1','Antennas_Cell2','Antennas_Cell3'],axis=1,inplace=True)


cell_complete['es_total'] = cell_complete['ESMode1'] + cell_complete['ESMode2'] + cell_complete['ESMode3'] + cell_complete['ESMode4'] + cell_complete['ESMode5'] + cell_complete['ESMode6']
cell_complete['load_total'] = cell_complete['load'] + cell_complete['load_Cell1'].fillna(0) + cell_complete['load_Cell2'].fillna(0) + cell_complete['load_Cell3'].fillna(0)
#cell_complete['txpower_total'] = cell_complete['TXpower'] + cell_complete['TXpower_Cell1'].fillna(0) + cell_complete['TXpower_Cell2'].fillna(0) + cell_complete['TXpower_Cell3'].fillna(0)


cell_complete['es_cell1_total'] = cell_complete['ESMode1_Cell1'] + cell_complete['ESMode2_Cell1'] + cell_complete['ESMode3_Cell1'] + cell_complete['ESMode4_Cell1'] + cell_complete['ESMode5_Cell1'] + cell_complete['ESMode6_Cell1']
cell_complete['es_cell2_total'] = cell_complete['ESMode1_Cell2'] + cell_complete['ESMode2_Cell2'] + cell_complete['ESMode3_Cell2'] + cell_complete['ESMode4_Cell2'] + cell_complete['ESMode5_Cell2'] + cell_complete['ESMode6_Cell2']
#cell_complete['es_cell3_total'] = cell_complete['ESMode1_Cell3'] + cell_complete['ESMode2_Cell3'] + cell_complete['ESMode3_Cell3'] + cell_complete['ESMode4_Cell3'] + cell_complete['ESMode5_Cell3'] + cell_complete['ESMode6_Cell3']


#cell_complete['bandwidth_total'] = cell_complete['Bandwidth'] + cell_complete['Bandwidth_Cell1'].fillna(0)
#cell_complete['bandwidth_total'] = cell_complete['bandwidth_total'].astype(str)

#cell_complete['frequency_total'] = cell_complete['Frequency'] + cell_complete['Frequency_Cell1'].fillna(0)

#cell_complete['load_sdev'] = np.std(cell_complete[['load','load_Cell1','load_Cell2','load_Cell3']],axis=1)

#cell_complete['nonZero_esModes'] = cell_complete[['ESMode1','ESMode2','ESMode3','ESMode4']].gt(0).sum(axis=1)
#cell_complete['nonZero_esModes_shift1'] = cell_complete['nonZero_esModes'].shift(1)

#max_val = np.max([cell_complete['TXpower'], cell_complete['TXpower_Cell1'].fillna(0), cell_complete['TXpower_Cell2'].fillna(0), cell_complete['TXpower_Cell3'].fillna(0)],axis=0)
#min_val = np.min([cell_complete['TXpower'], cell_complete['TXpower_Cell1'].fillna(0), cell_complete['TXpower_Cell2'].fillna(0), cell_complete['TXpower_Cell3'].fillna(0)],axis=0)
#cell_complete['tx_range'] = max_val - min_val

max_val = np.max([cell_complete['Bandwidth'], cell_complete['Bandwidth_Cell1'].fillna(0), cell_complete['Bandwidth_Cell2'].fillna(0), cell_complete['Bandwidth_Cell3'].fillna(0)],axis=0)
min_val = np.min([cell_complete['Bandwidth'], cell_complete['Bandwidth_Cell1'].fillna(0), cell_complete['Bandwidth_Cell2'].fillna(0), cell_complete['Bandwidth_Cell3'].fillna(0)],axis=0)
cell_complete['bandwidth_range'] = max_val - min_val
#cell_complete['bandwidth_max'] = max_val

#max_val = np.max([cell_complete['Frequency'], cell_complete['Frequency_Cell1'].fillna(0), cell_complete['Frequency_Cell2'].fillna(0), cell_complete['Frequency_Cell3'].fillna(0)],axis=0)
#min_val = np.min([cell_complete['Frequency'], cell_complete['Frequency_Cell1'].fillna(0), cell_complete['Frequency_Cell2'].fillna(0), cell_complete['Frequency_Cell3'].fillna(0)],axis=0)
#cell_complete['Frequency_range'] = max_val - min_val


cell_complete['bs_num'] = cell_complete['BS'].str.replace('B_','').astype(int).astype(str)


bs_rows = cell_complete.groupby(['bs_num']).size().reset_index()
bs_rows = dict(zip(bs_rows['bs_num'],bs_rows[0]))
cell_complete['bs_readings'] = cell_complete['bs_num'].map(bs_rows)


cell_complete['config'] = cell_complete['num_cells'].astype(str) + '_' + cell_complete['Bandwidth'].astype(str) + '_' + cell_complete['Frequency'].astype(str) + '_' + cell_complete['antenna_total'].astype(str)


cell_complete = cell_complete.drop(['load','load_Cell1','ESMode4','ESMode4_Cell1','ESMode4_Cell2','ESMode4_Cell3'],axis=1)



# Time-Series Features

In [ ]:
# -------------------------------------
# TIME-SERIES FEATURES

# Note: we need to do it with the testing data because of the nature of samples
# they chose being in the middle of some training data

def ts_feature_creator(input_data, value_col, group_col, feature_type, time_period, metric, extra_suffix):

    # ******

    # input_data should be pandas series (I.e. the column you want to lag/roll/diff from your dataframe)
    # feature_type should be one of: 'difference', 'lag', 'rolling'
    # - difference will subtract rows based on the time period provided
    # - lag will shift previous rows
    # - rolling will get the metric value (mean, median, etc.) of the previous months given in the time_period
    # avoid_cols should be a list of any columns to avoid, if none then give an empty list
    # time_period should be an integer that indicates how far back to compare against
    # metric should be a string that indicates the metric to calculate (this is for rolling features only)
    # - provide false if you are not using a rolling function
    # extra_suffix is for when you don't want to overwrite use: '' or 'string'

    # ******

    if feature_type == 'difference':
        output_feats = input_data.groupby(group_col)[value_col].diff(periods=time_period)
        output_feats = output_feats.add_suffix('_'+str(time_period)+'H_DIFF'+extra_suffix)
        output_feats.name = value_col+'_'+str(time_period)+'H_DIFF'+extra_suffix

    elif feature_type == 'dividing':
        output_feats = input_data.groupby(group_col)[value_col].div(input_data.groupby(group_col)[value_col].shift(time_period))
        output_feats = output_feats.add_suffix('_'+str(time_period)+'H_DIV'+extra_suffix)
        output_feats.name = value_col+'_'+str(time_period)+'H_DIV'+extra_suffix

    elif feature_type == 'percent_change':
        output_feats = input_data.groupby(group_col)[value_col].diff(periods=time_period).div(input_data.groupby(group_col)[value_col].shift(time_period))
        output_feats = output_feats.add_suffix('_'+str(time_period)+'H_PC'+extra_suffix)
        output_feats.name = value_col+'_'+str(time_period)+'H_PC'+extra_suffix

    elif feature_type == 'lag':
        output_feats = input_data.groupby(group_col)[value_col].shift(time_period)
        output_feats = output_feats.add_suffix('_'+str(time_period)+'H_LAG'+extra_suffix)
        output_feats.name = value_col+'_'+str(time_period)+'H_LAG'+extra_suffix

    elif feature_type == 'rolling':
        if metric == 'std':
            output_feats = input_data.groupby(group_col)[value_col].rolling(time_period,center=True).std()
        elif metric == 'mean':
            output_feats = input_data.groupby(group_col)[value_col].rolling(time_period,center=True).mean()
        elif metric == 'median':
            output_feats = input_data.groupby(group_col)[value_col].rolling(time_period,center=True).median()
        elif metric == 'min':
            output_feats = input_data.groupby(group_col)[value_col].rolling(time_period,center=True).min()
        elif metric == 'max':
            output_feats = input_data.groupby(group_col)[value_col].rolling(time_period,center=True).max()
        elif metric == 'var':
            output_feats = input_data.groupby(group_col)[value_col].rolling(time_period,center=True).var()
        elif metric == 'sum':
            output_feats = input_data.groupby(group_col)[value_col].rolling(time_period,center=True).sum()

        output_feats = output_feats.add_suffix('_'+str(time_period)+'H_roll'+metric.upper()+extra_suffix)
        output_feats.name = value_col+'_'+str(time_period)+'H_roll'+metric.upper()+extra_suffix

    elif feature_type == 'expanding':
        if metric == 'std':
            output_feats = input_data.groupby(group_col)[value_col].expanding(time_period).std()
        elif metric == 'mean':
            output_feats = input_data.groupby(group_col)[value_col].expanding(time_period).mean()
        elif metric == 'median':
            output_feats = input_data.groupby(group_col)[value_col].expanding(time_period).median()
        elif metric == 'min':
            output_feats = input_data.groupby(group_col)[value_col].expanding(time_period).min()
        elif metric == 'max':
            output_feats = input_data.groupby(group_col)[value_col].expanding(time_period).max()
        elif metric == 'var':
            output_feats = input_data.groupby(group_col)[value_col].expanding(time_period).var()
        elif metric == 'sum':
            output_feats = input_data.groupby(group_col)[value_col].expanding(time_period).sum()

        output_feats = output_feats.add_suffix('_'+str(time_period)+'H_exp'+metric.upper()+extra_suffix)
        output_feats.name = value_col+'_'+str(time_period)+'H_exp'+metric.upper()+extra_suffix

    #output_feats = output_feats.dropna().reset_index(drop=True)

    if value_col != 'Energy':
        output_feats = output_feats.fillna(method='bfill')

    output_data = input_data.copy()
    output_data[output_feats.name] = output_feats.values

    return output_data

In [ ]:
# GENERATE TSFRESH FEATURES

def generate_ts_feats(input_data, value_col, suffix):

    ts_features = pd.DataFrame()

    for bs_now in input_data['BS'].unique():
        sample_bs = input_data[input_data['BS'] == bs_now]

        #ts_output = tsfresh.extract_features(sample_bs[['Time_BS','load_total']].fillna(0), column_id='Time_BS')
        abs_energy = pd.Series(tsfresh.feature_extraction.feature_calculators.abs_energy(sample_bs[value_col]),name='abs_energy'+suffix)
        abs_max = pd.Series(tsfresh.feature_extraction.feature_calculators.absolute_maximum(sample_bs[value_col]),name='abs_max'+suffix)
        abs_sumChanges = pd.Series(tsfresh.feature_extraction.feature_calculators.absolute_sum_of_changes(sample_bs[value_col]),name='abs_sumChanges'+suffix)
        benford_corr = pd.Series(tsfresh.feature_extraction.feature_calculators.benford_correlation(sample_bs[value_col]),name='benford_corr'+suffix)
        c3 = pd.Series(tsfresh.feature_extraction.feature_calculators.c3(sample_bs[value_col], 1),name='c3'+suffix)

        countAboveMean = pd.Series(tsfresh.feature_extraction.feature_calculators.count_above_mean(sample_bs[value_col]),name='aboveMean'+suffix)
        countBelowMean = pd.Series(tsfresh.feature_extraction.feature_calculators.count_below_mean(sample_bs[value_col]),name='belowMean'+suffix)

        firstMax = pd.Series(tsfresh.feature_extraction.feature_calculators.first_location_of_maximum(sample_bs[value_col]),name='firstMax'+suffix)
        firstMin = pd.Series(tsfresh.feature_extraction.feature_calculators.first_location_of_minimum(sample_bs[value_col]),name='firstMin'+suffix)
        lastMax = pd.Series(tsfresh.feature_extraction.feature_calculators.last_location_of_maximum(sample_bs[value_col]),name='lastMax'+suffix)
        lastMin = pd.Series(tsfresh.feature_extraction.feature_calculators.last_location_of_minimum(sample_bs[value_col]),name='lastMin'+suffix)

        duplicateMax = pd.Series(tsfresh.feature_extraction.feature_calculators.has_duplicate_max(sample_bs[value_col]),name='duplicateMax'+suffix)
        duplicateMin = pd.Series(tsfresh.feature_extraction.feature_calculators.has_duplicate_min(sample_bs[value_col]),name='duplicateMin'+suffix)

        kurtosis = pd.Series(tsfresh.feature_extraction.feature_calculators.kurtosis(sample_bs[value_col]),name='kurtosis'+suffix)
        skewness = pd.Series(tsfresh.feature_extraction.feature_calculators.skewness(sample_bs[value_col]),name='skewness'+suffix)
        largeStd = pd.Series(tsfresh.feature_extraction.feature_calculators.large_standard_deviation(sample_bs[value_col],2),name='largeSTD'+suffix)

        longestAboveMean = pd.Series(tsfresh.feature_extraction.feature_calculators.longest_strike_above_mean(sample_bs[value_col]),name='longAboveMean'+suffix)
        longestBelowMean = pd.Series(tsfresh.feature_extraction.feature_calculators.longest_strike_below_mean(sample_bs[value_col]),name='longBelowMean'+suffix)

        meanAbsChange = pd.Series(tsfresh.feature_extraction.feature_calculators.mean_abs_change(sample_bs[value_col]),name='meanAbsChange'+suffix)
        meanChange = pd.Series(tsfresh.feature_extraction.feature_calculators.mean_change(sample_bs[value_col]),name='meanChange'+suffix)
        mean2ndDeriv = pd.Series(tsfresh.feature_extraction.feature_calculators.mean_second_derivative_central(sample_bs[value_col]),name='mean2ndDeriv'+suffix)
        percRepeatedRows = pd.Series(tsfresh.feature_extraction.feature_calculators.percentage_of_reoccurring_datapoints_to_all_datapoints(sample_bs[value_col]),name='percRepeatedRows'+suffix)
        percRepeatedVals = pd.Series(tsfresh.feature_extraction.feature_calculators.percentage_of_reoccurring_values_to_all_values(sample_bs[value_col]),name='percRepeatedVals'+suffix)

        ratioBeyondSigma = pd.Series(tsfresh.feature_extraction.feature_calculators.ratio_beyond_r_sigma(sample_bs[value_col],2),name='ratioBeyondSig'+suffix)
        ratioValueLen = pd.Series(tsfresh.feature_extraction.feature_calculators.ratio_value_number_to_time_series_length(sample_bs[value_col]),name='ratioValueLen'+suffix)

        rootMeanSq = pd.Series(tsfresh.feature_extraction.feature_calculators.root_mean_square(sample_bs[value_col]),name='rootMeanSq'+suffix)
        sumRepeatedRows = pd.Series(tsfresh.feature_extraction.feature_calculators.sum_of_reoccurring_data_points(sample_bs[value_col]),name='sumRepeatedRows'+suffix)
        sumRepeatedVals = pd.Series(tsfresh.feature_extraction.feature_calculators.sum_of_reoccurring_values(sample_bs[value_col]),name='sumRepeatedVals'+suffix)
        sumVals = pd.Series(tsfresh.feature_extraction.feature_calculators.sum_values(sample_bs[value_col]),name='sumVals'+suffix)

        varCoeff = pd.Series(tsfresh.feature_extraction.feature_calculators.variation_coefficient(sample_bs[value_col]),name='varCoeff'+suffix)

        #agg_autocorr = tsfresh.feature_extraction.feature_calculators.agg_autocorrelation(sample_bs['load_total'])
        #agg_linearTrend = tsfresh.feature_extraction.feature_calculators.agg_linear_trend(sample_bs['load_total'])
        #apx_entropy = tsfresh.feature_extraction.feature_calculators.approximate_entropy(sample_bs['load_total'])
        #ar_coeff = tsfresh.feature_extraction.feature_calculators.ar_coefficient(sample_bs['load_total'])

        temp_features = pd.DataFrame({

            "BS": [bs_now],
            abs_energy.name: abs_energy,
            abs_max.name: abs_max,
            abs_sumChanges.name: abs_sumChanges,
            benford_corr.name: benford_corr,
            c3.name: c3,

            countAboveMean.name: countAboveMean,
            countBelowMean.name: countBelowMean,
            firstMax.name: firstMax,
            firstMin.name: firstMin,
            lastMax.name: lastMax,
            lastMin.name: lastMin,
            duplicateMax.name: duplicateMax,
            duplicateMin.name: duplicateMin,
            kurtosis.name: kurtosis,
            skewness.name: skewness,
            largeStd.name: largeStd,

            longestAboveMean.name: longestAboveMean,
            longestBelowMean.name: longestBelowMean,
            meanAbsChange.name: meanAbsChange,
            meanChange.name: meanChange,
            mean2ndDeriv.name: mean2ndDeriv,
            percRepeatedRows.name: percRepeatedRows,
            percRepeatedVals.name: percRepeatedVals,

            ratioBeyondSigma.name: ratioBeyondSigma,
            ratioValueLen.name: ratioValueLen,

            rootMeanSq.name: rootMeanSq,
            sumRepeatedRows.name: sumRepeatedRows,
            sumRepeatedVals.name: sumRepeatedVals,
            sumVals.name: sumVals,

            varCoeff.name: varCoeff,

            })

        ts_features = pd.concat([ts_features,temp_features],axis=0)

    return ts_features


# Pipeline Functions

In [ ]:
def agg_features(train_set, test_set, group_cols, value_col, metric):

    if metric == 'mean':
        agg_data = train_set.groupby(group_cols)[value_col].mean().reset_index()
    elif metric == 'median':
        agg_data = train_set.groupby(group_cols)[value_col].median().reset_index()
    elif metric == 'max':
        agg_data = train_set.groupby(group_cols)[value_col].max().reset_index()
    elif metric == 'min':
        agg_data = train_set.groupby(group_cols)[value_col].min().reset_index()
    elif metric == 'std':
        agg_data = train_set.groupby(group_cols)[value_col].std().reset_index()

    col_names = group_cols.copy()
    col_names.append('_'.join(group_cols)+'_'+value_col+'_'+metric)
    agg_data.columns = col_names

    train_set = train_set.merge(agg_data, how='left', left_on=group_cols, right_on=group_cols)
    test_set = test_set.merge(agg_data, how='left', left_on=group_cols, right_on=group_cols)

    return train_set, test_set


def run_feature_function(train_set, test_set):

    #train_set, test_set = agg_features(train_set, test_set, ['Time','antenna_total','num_cells'], 'load_total', 'mean')
    #train_set, test_set = agg_features(train_set, test_set, ['Time'], 'load_total', 'mean')
    #train_set, test_set = agg_features(train_set, test_set, ['Time','num_cells'], 'Energy_1H_LAG', 'mean')
    #train_set, test_set = agg_features(train_set, test_set, ['hour','day_of_week'], 'load_total', 'std')
    #train_set, test_set = agg_features(train_set, test_set, ['hour','day_of_week'], 'es_total', 'mean')
    #train_set, test_set = agg_features(train_set, test_set, ['hour','day_of_week'], 'es_total', 'std')

    train_set, test_set = agg_features(train_set, test_set, ['hour'], 'load_total', 'mean')
    train_set, test_set = agg_features(train_set, test_set, ['hour'], 'es_total', 'mean')

    #train_set, test_set = agg_features(train_set, test_set, ['BS'], 'Energy_1H_LAG', 'mean')
    train_set, test_set = agg_features(train_set, test_set, ['BS'], 'load_total', 'mean')
    train_set, test_set = agg_features(train_set, test_set, ['BS'], 'es_total', 'mean')

    #train_set, test_set = agg_features(train_set, test_set, ['hour','num_cells'], 'load_total', 'mean')
    #train_set, test_set = agg_features(train_set, test_set, ['hour','num_cells'], 'es_total', 'mean')

    #train_set, test_set = agg_features(train_set, test_set, ['day_of_week','cluster_number'], 'Energy_1H_LAG', 'mean')

    #train_set, test_set = agg_features(train_set, test_set, ['hour','num_cells'], 'Energy_2H_LAG', 'mean')
    #train_set, test_set = agg_features(train_set, test_set, ['hour','num_cells'], 'load_total_1H_LAG', 'mean')

    #train_set, test_set = agg_features(train_set, test_set, ['Mode','day_of_week'], 'Energy_1H_LAG', 'mean')
    #train_set, test_set = agg_features(train_set, test_set, ['Mode','day_of_week'], 'Energy_2H_LAG', 'mean')

    return train_set, test_set



def normalized_features(train_set, test_set, value_col):

    greatest_val = train_set[value_col].max()

    train_set[value_col+'_percentMax'] = train_set[value_col] / greatest_val
    test_set[value_col+'_percentMax'] = test_set[value_col] / greatest_val

    return train_set, test_set


# -------------------------------------
# IMPUTATION STRATEGIES

def int_noneTag_imputation(train_set, test_set, num_fill):

    train_set = pd.concat([train_set.select_dtypes('number').fillna(num_fill),
                           train_set.select_dtypes('object').fillna('missing_data_here_eh')],axis=1)
    test_set = pd.concat([test_set.select_dtypes('number').fillna(num_fill),
                          test_set.select_dtypes('object').fillna('missing_data_here_eh')],axis=1)

    return train_set, test_set


def mean_none_imputation(train_set, test_set):

    # Now for the other numeric columns simply impute the mean
    numeric_cols = list(train_set.select_dtypes('number').columns.values)

    num_imputer = SimpleImputer(strategy='mean', keep_empty_features=True).fit(train_set[numeric_cols])
    train_nums = pd.DataFrame(num_imputer.transform(train_set[numeric_cols]), columns=train_set[numeric_cols].columns, index=train_set.index)
    test_nums = pd.DataFrame(num_imputer.transform(test_set[numeric_cols]), columns=test_set[numeric_cols].columns, index=test_set.index)

    # Now simply fill with 'None' for the categorical features
    train_cats = train_set.select_dtypes('object')
    test_cats = test_set.select_dtypes('object')

    train_cats = train_cats.fillna(value='missing_data_here_eh')
    test_cats = test_cats.fillna(value='missing_data_here_eh')

    train_set = pd.concat([train_nums,train_cats],axis=1)
    test_set = pd.concat([test_nums,test_cats],axis=1)

    return train_set, test_set


def median_none_imputation(train_set,test_set):

    # Now for the other numeric columns simply impute the mean
    numeric_cols = list(train_set.select_dtypes('number').columns)

    num_imputer = SimpleImputer(strategy='median', keep_empty_features=True).fit(train_set[numeric_cols])
    train_nums = pd.DataFrame(num_imputer.transform(train_set[numeric_cols]), columns = train_set[numeric_cols].columns, index=train_set.index)
    test_nums = pd.DataFrame(num_imputer.transform(test_set[numeric_cols]), columns = test_set[numeric_cols].columns, index=test_set.index)

    # Now simply fill with 'None' for the categorical features
    train_cats = train_set.select_dtypes('object').fillna('missing_data_here_eh')
    test_cats = test_set.select_dtypes('object').fillna('missing_data_here_eh')

    train_set = pd.concat([train_nums,train_cats],axis=1)
    test_set = pd.concat([test_nums,test_cats],axis=1)

    return train_set, test_set



def knn_noneTag_imputation(train_set,test_set):

    numeric_cols = list(train_set.select_dtypes('number').columns.values)

    # KNN works best with normalized data so we will normalize, fit to KNN imputer, and then reverse the normalization
    scaler = StandardScaler()
    scaled_train_nums = pd.DataFrame(scaler.fit_transform(train_set[numeric_cols]), columns = train_set[numeric_cols].columns, index=train_set.index)
    scaled_test_nums = pd.DataFrame(scaler.transform(test_set[numeric_cols]), columns = test_set[numeric_cols].columns, index=test_set.index)

    # Fill missing numeric with nearest neighbour values
    knn_imputer = KNNImputer(n_neighbors=30, keep_empty_features=True).fit(scaled_train_nums)
    train_num = pd.DataFrame(knn_imputer.transform(scaled_train_nums), columns = scaled_train_nums.columns, index=train_set.index)
    test_num = pd.DataFrame(knn_imputer.transform(scaled_test_nums), columns = scaled_test_nums.columns, index=test_set.index)

    # Now reverse scaling using KNN output
    train_num = pd.DataFrame(scaler.inverse_transform(train_num), columns = train_num.columns, index=train_set.index)
    test_num = pd.DataFrame(scaler.inverse_transform(test_num), columns = test_num.columns, index=test_set.index)

    # Now simply fill with 'None' for the categorical features
    train_cats = train_set.select_dtypes('object').fillna('missing_data_here_eh')
    test_cats = test_set.select_dtypes('object').fillna('missing_data_here_eh')


    train_set = pd.concat([train_num,train_cats],axis=1)
    test_set = pd.concat([test_num,test_cats],axis=1)

    return train_set, test_set


# -------------------------------------
# OUTLIER CAPPING


def category_capper(train_set, test_set):

  capper = RareLabelEncoder(n_categories=30, tol=0.005, replace_with='rarities',
                            #variables=[]
                            )

  capper.fit(train_set)
  train_set = capper.transform(train_set)
  test_set = capper.transform(test_set)

  train_output, test_output = train_set, test_set

  return train_output, test_output


def outlier_capper(train_set, test_set):

  capper = Winsorizer(capping_method='gaussian',
                      tail='right', fold=2.5, add_indicators=False,
                      variables=['load'],
                      missing_values='raise'
                      )

  capper.fit(train_set)
  train_set = capper.transform(train_set)
  test_set = capper.transform(test_set)

  train_output, test_output = train_set, test_set

  return train_output, test_output




# -------------------------------------
# DUMMY ENCODER FOR CATEGORICAL VARIABLES

def dummy_encoder(x_trainData, x_testData):

    # Create flag before concat
    x_trainData['trainTest'] = 'train'
    x_testData['trainTest'] = 'test'

    # Concat into one
    input_data = pd.concat([x_trainData,x_testData],axis=0)

    # Loop through object columns and transform to dummy variable
    collector = pd.DataFrame()
    for col in input_data.select_dtypes('object'):
        if col != 'trainTest':
            col_dummies = pd.get_dummies(input_data[col], drop_first=True, prefix=col, prefix_sep='_')
            collector = pd.concat([collector, col_dummies], axis=1)

    # Combine encoded object data with numeric data
    output_data = pd.concat([input_data.select_dtypes(['number']),collector,input_data['trainTest']],axis=1)

    # Split data up and drop flag column created earlier
    x_trainOutput = output_data[output_data['trainTest']=='train']
    x_trainOutput.drop(['trainTest'],axis=1,inplace=True)

    x_testOutput = output_data[output_data['trainTest']=='test']
    x_testOutput.drop(['trainTest'],axis=1,inplace=True)

    return x_trainOutput, x_testOutput


# -------------------------------------
# FREQUENCY-BASED ORDINAL ENCODER FOR CATEGORICAL VARIABLES
# note: this is to be used on remaining categorical features AFTER domain knowledge is used in previous ordinal encoder

def freqOrd_encoder(x_trainCV, x_valCV, y_trainCV):

  for col in x_trainCV.select_dtypes('object'):

    # Get the frequencies of each category in the column
    group_counts = x_trainCV.groupby(col).size().reset_index(name='frequency').sort_values(by=['frequency'],ascending=False)

    # Use the frequencies as our basis for ordinal encoding
    group_counts = group_counts.reset_index().reset_index()
    group_counts['ord_value'] = group_counts['level_0'] + 1

    # create dictionary to map/overwrite categories with new value
    group_dict = dict(zip(group_counts[col],group_counts['ord_value']))
    group_dict['missing'] = -1

    # Map new values to training and testing
    x_trainCV[col] = x_trainCV[col].map(group_dict)
    x_valCV[col] = x_valCV[col].map(group_dict)

    # Some values in testing may not be found in training resulting in NaN so fill those with -2
    x_valCV[col] = x_valCV[col].fillna(-2)

  return x_trainCV, x_valCV




# -------------------------------------
# COUNT ENCODER FOR CATEGORICAL VARIABLES

def count_encoder(x_trainCV, x_valCV, y_trainCV):

    for col in x_trainCV.select_dtypes('object'):
        cnt_encoder = count_encoder = CountEncoder()

        x_trainCV[col] = cnt_encoder.fit_transform(x_trainCV[col],y_trainCV)
        x_valCV[col] = cnt_encoder.transform(x_valCV[col])

    return x_trainCV, x_valCV


# -------------------------------------
# TARGET ENCODER FOR CATEGORICAL VARIABLES

def target_encoder(x_trainCV, x_valCV, y_trainCV):

    #if y_trainCV.name == 'h1n1_vaccine':
    #    smooth_value = 0.001
    #    min_leaf_value = 1
    #elif y_trainCV.name == 'seasonal_vaccine':
    #    smooth_value = 0.001
    #    min_leaf_value = 100

    smooth_value = 10.0
    min_leaf_value = 1

    #print(smooth_value,min_leaf_value)

    for col in x_trainCV.select_dtypes('object'):
        target_encoder = TargetEncoder(smoothing=smooth_value, min_samples_leaf=min_leaf_value)

        x_trainCV[col] = target_encoder.fit_transform(x_trainCV[col],y_trainCV)
        x_valCV[col] = target_encoder.transform(x_valCV[col])

    return x_trainCV, x_valCV

# -------------------------------------
# M-ESTIMATE (XAM) TARGET ENCODER FOR CATEGORICAL VARIABLES

def mest_target_encoder(x_trainCV, x_valCV, y_trainCV):


    for col in x_trainCV.select_dtypes('object'):
        mest_encoder = MEstimateEncoder()

        x_trainCV[col] = mest_encoder.fit_transform(x_trainCV[col],y_trainCV)
        x_valCV[col] = mest_encoder.transform(x_valCV[col])

    return x_trainCV, x_valCV


# -------------------------------------
# JAMES-STEIN TARGET ENCODER FOR CATEGORICAL VARIABLES

def js_target_encoder(x_trainCV, x_valCV, y_trainCV):

    for col in x_trainCV.select_dtypes('object'):
        js_encoder = JamesSteinEncoder()

        x_trainCV[col] = js_encoder.fit_transform(x_trainCV[col],y_trainCV)
        x_valCV[col] = js_encoder.transform(x_valCV[col])

    return x_trainCV, x_valCV


# -------------------------------------
# GLMM TARGET ENCODER FOR CATEGORICAL VARIABLES

def glmm_encoder(x_trainCV, x_valCV, y_trainCV):

    for col in x_trainCV.select_dtypes('object'):
        GLMM_encoder = GLMMEncoder(binomial_target=False)

        x_trainCV[col] = GLMM_encoder.fit_transform(x_trainCV[col],y_trainCV)
        x_valCV[col] = GLMM_encoder.transform(x_valCV[col])

    return x_trainCV, x_valCV


# -------------------------------------
# LEAVE-ONE-OUT TARGET ENCODER FOR CATEGORICAL VARIABLES

def loo_encoder(x_trainCV, x_valCV, y_trainCV):

    for col in x_trainCV.select_dtypes('object'):
        LOO_encoder = LeaveOneOutEncoder(sigma=0.05, random_state=117)

        x_trainCV[col] = LOO_encoder.fit_transform(x_trainCV[col],y_trainCV)
        x_valCV[col] = LOO_encoder.transform(x_valCV[col])

    return x_trainCV, x_valCV

# -------------------------------------
# CATBOOST TARGET ENCODER FOR CATEGORICAL VARIABLES

def catboost_encoder(x_trainCV, x_valCV, y_trainCV):

    for col in x_trainCV.select_dtypes('object'):
        catBoost_encoder = CatBoostEncoder(sigma=0.05, random_state=117)

        x_trainCV[col] = catBoost_encoder.fit_transform(x_trainCV[col],y_trainCV)
        x_valCV[col] = catBoost_encoder.transform(x_valCV[col])

    return x_trainCV, x_valCV






# -------------------------------------
# SCALING

def stand_scaling(train_set, test_set):

    # At this point all data will be numeric so no need to separate categorical features
    for col in train_set.select_dtypes('number'):
        if col not in ['Id','EJ']:

            stand_scaler = StandardScaler()
            train_set[[col]] = stand_scaler.fit_transform(train_set[[col]])
            test_set[[col]] = stand_scaler.transform(test_set[[col]])

    return train_set, test_set


def robust_scaling(train_set, test_set):

    # At this point all data will be numeric so no need to separate categorical features
    for col in train_set.select_dtypes('number'):
        if col not in ['Id','EJ']:
            if list(train_set[col].unique()) not in [[0,1],[1,0],[0],[1]]:
                rob_scaler = RobustScaler()
                train_set[[col]] = rob_scaler.fit_transform(train_set[[col]])
                test_set[[col]] = rob_scaler.transform(test_set[[col]])

    return train_set, test_set

def minmax_scaling(train_set, test_set):

    # At this point all data will be numeric so no need to separate categorical features
    for col in train_set.select_dtypes('number'):
        if col not in ['Id','EJ']:

            mm_scaler = MinMaxScaler()
            train_set[[col]] = mm_scaler.fit_transform(train_set[[col]])
            test_set[[col]] = mm_scaler.transform(test_set[[col]])

    return train_set, test_set




# -------------------------------------
# FINAL CLEAN

def cleanup(train_set, test_set):

    for col in ['Time','BS','Time_BS','train_flag','Mode','RUType']:
        if col in train_set.columns:
            train_set = train_set.drop([col],axis=1)
        if col in test_set.columns:
            test_set = test_set.drop([col],axis=1)


    # Convert object types to categorical
    # note: this is done for when using lightgbm default encoding, also helps with memory
    cat_cols = list(train_set.select_dtypes('object'))
    for c in cat_cols:
      train_set[c] = train_set[c].astype('category')
      test_set[c] = test_set[c].astype('category')


    return train_set, test_set





# -------------------------------------
# PIPELINES CREATOR


def pipeline_creator(training_data,testing_data,y_labels,
                     auto_style,
                     impute_style,
                     target_style,
                     cat_style,
                     scale_style):

    y_labels = y_labels['Energy']

    impute_options = {"int_None": int_noneTag_imputation,
                      "mean_None":mean_none_imputation,
                      "median_None":median_none_imputation,
                      "knn_None": knn_noneTag_imputation}

    catEncode_options = {"dummy": dummy_encoder,
                         "target": target_encoder,
                         "loo": loo_encoder,
                         "catboost": catboost_encoder,
                         "mest": mest_target_encoder,
                         "glmm": glmm_encoder,
                         "jamesStein": js_target_encoder,
                         "count": count_encoder,
                         #"poly": poly_encoder,
                         "freqOrd": freqOrd_encoder,
                         }

    scaling_options = {"standard": stand_scaling,
                       "robust": robust_scaling,
                       "minmax": minmax_scaling,
                       }

    # **************************
    # ***** Build Pipeline *****


    # --------------------
    # -- Auto features --
    if auto_style == True:
        train_, test_ = auto_features(training_data, testing_data)
    else:
        train_, test_ = training_data, testing_data


    # --------------------
    # -- Imputation --
    impute_strategy = impute_options.get(impute_style)
    if impute_style == 'int_None':
        train_, test_ = impute_strategy(train_, test_, 0)
    else:
        train_, test_ = impute_strategy(train_, test_)



    # --------------------------
    # -- Categorical Encoding --

    if target_style != False:

      target_encoding = list(target_style.keys())[0]
      columns_to_targetEncode = target_style[target_encoding]

      # Separate necessary columns, and transform ONLY THEM using target encoding style provided
      cat_strategy = catEncode_options.get(target_encoding)
      train_encoded, test_encoded = cat_strategy(train_[columns_to_targetEncode],
                                                 test_[columns_to_targetEncode], y_labels)
      train_other, test_other = train_.drop(columns_to_targetEncode,axis=1), test_.drop(columns_to_targetEncode,axis=1)

      # Bring data back together
      train_encoded = pd.concat([train_other,train_encoded],axis=1)
      test_encoded = pd.concat([test_other,test_encoded],axis=1)

      # We still have other categorical features to encode....
      # so finally lgbm encode (i.e. leave alone) any remaining categorical columns
      train_, test_ = train_encoded, test_encoded

      if cat_style == 'dummy':
          train_, test_ = dummy_encoder(train_, test_)
      elif cat_style == 'lgbm':
          train_, test_ = train_, test_
      elif cat_style == False:
          train_, test_ = train_, test_



    # if equals zero then we simply encode ALL categorical features using the encoder provided
    elif target_style == False:
      cat_strategy = catEncode_options.get(cat_style)

      if cat_style == 'dummy':
          train_, test_ = dummy_encoder(train_, test_)
      elif cat_style == 'lgbm':
          train_, test_ = train_, test_
      else:
          train_, test_ = cat_strategy(train_, test_, y_labels)



    # --------------------
    # -- Scaling --
    if scale_style != False:
        scaling_strategy = scaling_options.get(scale_style)
        train_, test_ = scaling_strategy(train_, test_)
    else:
        train_, test_ = train_, test_


    # --------------------
    # -- Final Cleaning --
    train_, test_ = cleanup(train_, test_)


    # -----------------
    # -- Return Data --
    return train_, test_





# -------------------------------------
# PIPELINES

# **********************************************************
# Create a dictionary of pipelines to iterate through
# and use the key values within the optuna function
# -----------------------


pipelines_to_build = {

    #"meanNone_js": ['mean_None', # imputation
    #             {'jamesStein':['EJ']}, # target encoding style and columns
    #             False, # categorical encoding of any remaining category features
    #             False, # scaling type
    #              ],

    #"meanNone_dummy": [False, 'mean_None', False, 'dummy', False],
    #"meanNone_freqOrd": [False, 'mean_None', False, 'freqOrd', False],
    "intNone_dummy": [False, 'int_None', False, 'dummy', False],
    "intNone_dummy_rob": [False, 'int_None', False, 'dummy', 'robust'],
    #"intNone_lgb": [False, 'int_None', False, 'lgbm', False],
    #"intNone_js_dummy": [False, 'int_None', {'jamesStein':['RUType','Mode']}, 'dummy', False],
    #"intNone_cat_dummy": [False, 'int_None', {'catboost':['mode_rutype','RUType']}, 'dummy', False],
    #"intNone_loo_rob": [False, 'int_None', {'loo':['EJ']}, 'dummy', 'robust'],
    #"meanNone_loo_rob": [False, 'mean_None', {'loo':['EJ']}, 'dummy', 'robust'],
    #"meanNone_catb_rob": [False, 'mean_None', {'catboost':['EJ']}, 'dummy', 'robust'],
    #"meanNone_dummy_std": [False, 'mean_None', False, 'dummy', 'standard'],
    #"meanNone_dummy_rob": [False, 'mean_None', False, 'dummy', 'robust'],
    #"meanNone_dummy_minmax": [False, 'mean_None', False, 'dummy', 'minmax'],
    #"meanNone_freqOrd_rob": [False, 'mean_None', False, 'freqOrd', 'robust'],
    #"medianNone_freqOrd_std": [False, 'median_None', False, 'freqOrd', 'standard'],
    #"medianNone_freqOrd_rob": [False, 'median_None', False, 'freqOrd', 'robust'],
    #"medianNone_dummy_std": [False, 'median_None', False, 'dummy', 'standard'],
    #"medianNone_dummy_rob": [False, 'median_None', False, 'dummy', 'robust'],
    #"medianNone_dummy": [False,'median_None', False, 'dummy', False],
    #"medianNone_freqOrd": [False, 'median_None', False, 'freqOrd', False],
    #"auto_medianNone_dummy": [True,'median_None', False, 'dummy', False],
    #"medianNone_catb": [False, 'median_None', {'catboost':['Mode']}, 'dummy', False],
    #"medianNone_mest_rob": [False,'median_None', {'mest':['EJ']}, 'dummy', 'robust'],
    #"medianNone_targ_rob": [False,'median_None', {'target':['EJ']}, 'dummy', 'robust'],
    #"auto_medianNone_dummy_rob": [True,'median_None', False, 'dummy', 'robust'],
    #"knnNone_dummy_rob": [False, 'knn_None', False, 'dummy', 'robust'],
    #"knnNone_dummy": [False, 'knn_None', False, 'dummy', False],
    #"auto_intNone_dummy_rob": [True,'int_None', False, 'dummy', 'robust'],

    }



# Data Transformation

In [ ]:

ts_load_feats = generate_ts_feats(cell_complete, 'load_total', '_load')
#ts_es_feats = generate_ts_feats(cell_complete, 'es_total', '_load')
cell_complete = cell_complete.merge(ts_load_feats, how='left', left_on=['BS'], right_on=['BS'])
#cell_complete = cell_complete.merge(ts_es_feats, how='left', left_on=['BS'], right_on=['BS'])

cell_transform = ts_feature_creator(cell_complete, 'load_total', 'BS', 'difference', 1, False, '')
#cell_transform = ts_feature_creator(cell_complete, 'load_total', 'BS', 'percent_change', 1, False, '')

cell_transform = ts_feature_creator(cell_complete, 'load_total', 'BS', 'lag', 1, False, '')
cell_transform = ts_feature_creator(cell_transform, 'load_total', 'BS', 'lag', 2, False, '')
cell_transform = ts_feature_creator(cell_transform, 'load_total', 'BS', 'lag', 3, False, '')
cell_transform = ts_feature_creator(cell_transform, 'load_total', 'BS', 'lag', 4, False, '')

#cell_transform = ts_feature_creator(cell_transform, 'es_total', 'BS', 'lag', 1, False, '')
#cell_transform = ts_feature_creator(cell_transform, 'es_total', 'BS', 'lag', 2, False, '')
#cell_transform = ts_feature_creator(cell_transform, 'es_total', 'BS', 'lag', 3, False, '')
#cell_transform = ts_feature_creator(cell_transform, 'es_total', 'BS', 'lag', 4, False, '')


#cell_transform = ts_feature_creator(cell_transform, 'load_total', 'BS', 'rolling', 3, 'mean', '')
cell_transform = ts_feature_creator(cell_transform, 'load_total', 'BS', 'rolling', 3, 'std', '')
#cell_transform = ts_feature_creator(cell_transform, 'es_total', 'BS', 'rolling', 3, 'mean', '')
cell_transform = ts_feature_creator(cell_transform, 'es_total', 'BS', 'rolling', 3, 'std', '')


#cell_transform = ts_feature_creator(cell_transform, 'es_total', ['BS','hour'], 'lag', 1, False, '_byHour')
#cell_transform = ts_feature_creator(cell_transform, 'load_total', ['BS','hour'], 'lag', 1, False, '_byHour')
#cell_transform = ts_feature_creator(cell_transform, 'load', ['BS','hour'], 'lag', 1, False, '_byHour')



# -------------------------------------
# TARGET TIME-SERIES FEATURES

cell_transform = cell_transform.merge(train_labels[['Time_BS','Energy']], how='left', left_on=['Time_BS'], right_on=['Time_BS'])

cell_transform = ts_feature_creator(cell_transform, 'Energy', 'BS', 'rolling', 2, 'mean', '')
#cell_transform = ts_feature_creator(cell_transform, 'Energy', 'BS', 'rolling', 3, 'std', '')

cell_transform = ts_feature_creator(cell_transform, 'Energy', 'BS', 'lag', 1, False, '')
cell_transform = ts_feature_creator(cell_transform, 'Energy', 'BS', 'lag', 2, False, '')
#cell_transform = ts_feature_creator(cell_transform, 'Energy', 'BS', 'lag', 3, False, '')

cell_transform = ts_feature_creator(cell_transform, 'Energy', ['BS','hour'], 'lag', 1, False, '_byHour')
#cell_transform = ts_feature_creator(cell_transform, 'Energy', ['BS','hour'], 'lag', 2, False, '_byHour')
#cell_transform = ts_feature_creator(cell_transform, 'Energy', ['BS','hour'], 'lag', 3, False, '_byHour')
#cell_transform = ts_feature_creator(cell_transform, 'Energy', ['BS','hour'], 'lag', 4, False, '_byHour')
#cell_transform = ts_feature_creator(cell_transform, 'Energy', ['BS','hour'], 'rolling', 2, 'mean', '_byHour')
#cell_transform = ts_feature_creator(cell_transform, 'Energy', 'BS', 'percent_change', 1, False, '')

# --------

# smooth mode 1 labels
def smooth_mode1_energy(row):
    if row['Time'] < pd.Timestamp('2023-01-02 04:00:00') and row['Mode'] == 'Mode1':
        return max(row['Energy'], 20)
    else:
        return row['Energy']
#cell_transform['Energy'] = cell_transform.apply(smooth_mode1_energy, axis=1)

cell_transform['Energy'] = np.where((cell_transform['train_flag']==1) & (cell_transform['Energy'] < 20) & (cell_transform['Mode']=='Mode1'), 20, cell_transform['Energy'])
train_labels['Energy'] = cell_transform[cell_transform['train_flag']==1]['Energy'].values


cell_transform = cell_transform.drop(['Energy',
#                                      #'Energy_2H_LAG_fill'
                                      ],axis=1)


In [ ]:
train = cell_transform[cell_transform['train_flag']==1].drop(['train_flag'],axis=1)
test = cell_transform[cell_transform['train_flag']==0].drop(['train_flag'],axis=1)

# Reset indices to make sure they match
train = train.reset_index(drop=True)
train_labels = train_labels.reset_index(drop=True)

# Model Evaluation

In [ ]:
# -------------------------------------
# WMAPE COMPETITION METRIC

def wmape_metric(true_values, pred_values, weights):

    true_values = true_values.reset_index(drop=True)
    pred_values = pred_values.reset_index(drop=True)
    weights = weights.reset_index(drop=True)


    numer = abs(true_values - pred_values) * weights
    denom = abs(true_values) * weights
    wmape_output = sum(numer)/sum(denom)


    # -----------------------------------
    # FROM INDEED ARTICLE

    # Get absolute percentage change for each prediction
    #pc = (abs(y_val['Energy'] - cv_preds) / y_val['Energy']) * 100

    # Get weights of each sample by dividing by actuals
    #weights = pc / y_val['Energy']

    # Now divide weights by
    #wmape_output = sum(weights) / sum(pc)
    # -----------------------------------

    return wmape_output

In [ ]:
# -------------------------------------
# MODEL EVALUATION


results = pd.DataFrame()

for pipeline_name in ['intNone_dummy']:

    pipeline_inputs = pipelines_to_build.get(pipeline_name)

    autofeat_style = pipeline_inputs[0]
    imputation_style = pipeline_inputs[1]
    target_input = pipeline_inputs[2]
    categorical_style = pipeline_inputs[3]
    scaling_style = pipeline_inputs[4]

    # Storage for predication analysis later on
    preds_eda=[]
    cv_importances=[]

    # Random states
    rands = [92,9]

    # Establish group k fold
    splits=10
    #folder = GroupKFold(n_splits=splits)
    folder = KFold(n_splits=splits)

    mae_scores = []
    wmape_scores = []

    # Loop through folds for each question
    #for i, (train_index, test_index) in enumerate(folder.split(X=train, groups=train['BS'])):
    for i, (train_index, test_index) in enumerate(folder.split(X=train, y=train_labels)):

        #print('processing fold:',i)

        # Separate data into fit and validation
        X_fit, X_val = train.iloc[train_index], train.iloc[test_index]
        y_fit, y_val = train_labels.iloc[train_index], train_labels.iloc[test_index]



        # -------------------------------------------------
        # IN-LOOP FEATURE ENGINEERING TO HELP AVOID LEAKAGE

        # Create aggregate features here to prevent leakage

        X_fit, X_val = run_feature_function(X_fit, X_val)


        #X_fit, X_val = normalized_features(X_fit, X_val, 'Energy_1H_LAG')

        #X_fit, X_val = hour_features(X_fit, X_val, 'Energy_1H_LAG', ['hour','num_cells'])

        #X_fit['Energy'] = y_fit['Energy']
        #X_fit, X_val = hour_features(X_fit, X_val, 'Energy', ['BS'])
        #X_fit = X_fit.drop(['Energy'],axis=1)

        # -------------------------------------------------









        X_fit, X_val = X_fit.reset_index(drop=True), X_val.reset_index(drop=True)
        y_fit, y_val = y_fit.reset_index(drop=True), y_val.reset_index(drop=True)

        X_fit, X_val = X_fit.drop(['Time','BS','Time_BS'],axis=1), X_val.drop(['Time','BS','Time_BS'],axis=1)



        X_fit, X_val = pipeline_creator(X_fit, X_val, y_fit,
                                        auto_style=autofeat_style,
                                        impute_style=imputation_style,
                                        target_style=target_input,
                                        cat_style=categorical_style,
                                        scale_style=scaling_style,
                                           )


        # Fit a single model and make predictions
        #cv_model = lightgbm.LGBMRegressor(importance_type='gain')
        #cv_model.fit(X_fit, y_fit['Energy'])
        #cv_preds = cv_model.predict(X_val)

        # Calculate scoring metrics
        #mae_value = mean_absolute_error(y_val['Energy'], cv_preds)
        #mae_scores.append(mae_value)

        #wmape_value = wmape_metric(y_val['Energy'], cv_preds, y_val['w'])
        #wmape_scores.append(wmape_value)



        # Fit a model for each random state and average predictions
        states_preds = pd.DataFrame()
        mae_RS_scores = []
        wmape_RS_scores = []

        for rand in rands:
            cv_model = lightgbm.LGBMRegressor(importance_type='gain', seed=rand, verbose=-1)
            #cv_model = xgboost.XGBRegressor()
            #cv_model = CatBoostRegressor(verbose=False)

            cv_model.fit(X_fit, y_fit['Energy'])
            cv_preds = pd.Series(cv_model.predict(X_val))

            # use this for tabnet
            #cv_model = TabNetRegressor(verbose=0,seed=rand)
            #cv_model.fit(X_fit.to_numpy(), y_fit['Energy'].to_numpy().reshape(-1,1),
            #          eval_set=[(X_val.to_numpy(), y_val['Energy'].to_numpy().reshape(-1,1))],
            #          patience=20, max_epochs=100,
            #          eval_metric=['mae'])
            #cv_preds = pd.Series(cv_model.predict(X_val.to_numpy())[:,0])

            mae_value = mean_absolute_error(y_val['Energy'], cv_preds)
            mae_RS_scores.append(mae_value)

            wmape_value = wmape_metric(y_val['Energy'], cv_preds, y_val['w'])
            wmape_RS_scores.append(wmape_value)

            states_preds = pd.concat([states_preds,pd.Series(cv_preds,name=f's{rand}')],axis=1)



            # Store feature importance data to evaluate model
            feature_importance_mapping = dict(zip(X_fit.columns, cv_model.feature_importances_))
            feature_importance_mapping = pd.DataFrame.from_dict(feature_importance_mapping, orient='index', columns=['Value'])
            feature_importance_mapping = feature_importance_mapping.reset_index()
            feature_importance_mapping.columns = ['feature','importance']
            feature_importance_mapping["cv_seed"] = rand
            feature_importance_mapping["cv_fold"] = i
            feature_importance_mapping["pipeline"] = pipeline_name
            cv_importances.append(feature_importance_mapping)




        # Average random state predictions here, this can be used for analysis or
        # to get scores for the seed-averaged version of the model
        cv_preds = pd.Series(np.mean(states_preds,axis=1))

        # Average the SCORES from each random state model,
        # this way it truly acts as separate models as opposed to an ensemble
        mae_scores.append(np.mean(mae_RS_scores))
        wmape_scores.append(np.mean(wmape_RS_scores))


        #** ---------------------------------
        # PREDICTION/MODEL EDA


        X_val['predictions'] = cv_preds.values
        X_val['trues'] = y_val['Energy'].values
        X_val['diff'] = cv_preds.values - y_val['Energy'].values
        X_val['pipeline'] = pipeline_name
        #X_val['cv_seed'] = state
        #X_val['cv_fold'] = i
        preds_eda.append(X_val)

        #** ---------------------------------


    #print('MAE Average:',np.mean(mae_scores))
    #print('wMAPE Average:',np.mean(wmape_scores))

    pipeline_results = pd.DataFrame({"Model":[pipeline_name],
                                     "Splits":[splits],
                                     "MAE":[np.mean(mae_scores)],
                                     "wMAPE":[np.mean(wmape_scores)],
                                     "Std Dev MAE":[np.std(mae_scores)],
                                     "Std Dev wMAPE":[np.std(wmape_scores)],
                                     })

    results = pd.concat([results, pipeline_results], axis=0)

print(results.sort_values(by=['wMAPE'],ascending=True))

all_importances = pd.concat(cv_importances)
importance_summary = all_importances.groupby(['pipeline','feature'])['importance'].agg(["mean", "std"]).reset_index().sort_values("mean", ascending=False)


           Model  Splits       MAE     wMAPE  Std Dev MAE  Std Dev wMAPE
0  intNone_dummy      10  1.707442  0.060779     0.147868       0.005699


# Submission Creation

In [ ]:
#  ***** SINGLE MODEL SUBMISSION *****

#train_sub, test_sub = run_feature_function(train, test)
train_sub, test_sub = train.copy(), test.copy()

train_sub, test_sub = run_feature_function(train_sub, test_sub)

train_sub, test_sub = train_sub.drop(['Time','BS','Time_BS'],axis=1), test_sub.drop(['Time','BS','Time_BS'],axis=1)


pipeline_inputs = pipelines_to_build.get('intNone_dummy')
autofeat_style = pipeline_inputs[0]
imputation_style = pipeline_inputs[1]
target_input = pipeline_inputs[2]
categorical_style = pipeline_inputs[3]
scaling_style = pipeline_inputs[4]

train_sub, test_sub = pipeline_creator(train_sub, test_sub, train_labels,
                               auto_style=autofeat_style,
                               impute_style=imputation_style,
                               target_style=target_input,
                               cat_style=categorical_style,
                               scale_style=scaling_style,
                                           )


model_ = lightgbm.LGBMRegressor(importance_type='gain')
#model_ = xgboost.XGBRegressor()
#model_ = CatBoostRegressor(verbose=False)
#cv_model = TabNetRegressor(verbose=0,seed=42)
#cv_model.fit(X_fit, y_fit['Energy'],
#          eval_set=[(X_val, y_val['Energy'])],
#          patience=300, max_epochs=2000,
#          eval_metric=['rmse'])

model_.fit(train_sub, train_labels['Energy'])
model_preds = pd.Series(model_.predict(test_sub))

model_preds_df = pd.concat([test[['Time','BS']].reset_index(drop=True),model_preds],axis=1)



# -----------------------------
# SPECIAL PREDICTIONS

train_sub = train_sub.drop(columns=train_sub.filter(like='Energy').columns)
test_sub = test_sub.drop(columns=test_sub.filter(like='Energy').columns)
model_ = lightgbm.LGBMRegressor(importance_type='gain')
#model_ = LassoCV(cv=5, random_state=0)
model_.fit(train_sub, train_labels['Energy'])
extra_preds = pd.Series(model_.predict(test_sub))

extra_preds_df = pd.concat([test[['Time','BS']].reset_index(drop=True),extra_preds],axis=1)
extra_preds_df = extra_preds_df[extra_preds_df['BS'].isin(['B_13'])==True]


model_preds_df = model_preds_df.merge(extra_preds_df[['Time','BS', 0]], how='left',
                                      left_on=['Time','BS'], right_on=['Time','BS'])
model_preds_df['sub_preds'] = np.where(model_preds_df['0_y'].isnull()==False, model_preds_df['0_y'], model_preds_df['0_x'])

# -----------------------------



submission_file = zindi_sub.copy()
#submission_file['Energy'] = model_preds
submission_file['Energy'] = model_preds_df['sub_preds']
#submission_file.drop(['w'],axis=1,inplace=True)
submission_file.to_csv('/drive/My Drive/Colab Notebooks/Zindi - Huawei/submission_huawei_file.csv', index=False)

# Github Sync

In [ ]:
def safe_setup_github():
    """Safe setup that doesn't break the current directory"""
    import os
    import shutil
    from google.colab import userdata
    import getpass

    # Always start from /content
    os.chdir('/content')
    print(f"Working from: {os.getcwd()}")

    # Get GitHub token
    try:
        github_token = userdata.get('zindi_repo_git')
        print("Using token from Colab secrets")
    except:
        github_token = getpass.getpass('Enter your GitHub personal access token: ')

    # Check if repo exists and handle properly
    if os.path.exists('/content/zindi_repo'):
        print("Repository exists, checking if it's valid...")

        if os.path.exists('/content/zindi_repo/.git'):
            print("Valid git repository found")
            os.chdir('/content/zindi_repo')

            # below is new code
            # ✅ CRITICAL: PULL LATEST CHANGES FROM GITHUB
            print("🔄 Pulling latest changes from GitHub...")
            !git pull origin main

            # Ensure folder exists
            os.makedirs('huawei_networks', exist_ok=True)

            print("✅ Repository updated with latest changes!")
            return True
        else:
            print("Removing non-git folder...")
            # Change to safe directory BEFORE deleting
            os.chdir('/content')
            shutil.rmtree('/content/zindi_repo')

    # Fresh clone
    print("Cloning fresh repository...")
    repo_url = f"https://{github_token}@github.com/fitzpk/zindi_repo.git"
    !git clone {repo_url}

    # Navigate to repo
    os.chdir('/content/zindi_repo')

    # Create folder
    os.makedirs('huawei_networks', exist_ok=True)

    print("GitHub setup complete!")
    return True

# Run the safe setup
safe_setup_github()

Working from: /content
Using token from Colab secrets
Repository exists, checking if it's valid...
Valid git repository found
🔄 Pulling latest changes from GitHub...
From https://github.com/fitzpk/zindi_repo
 * branch            main       -> FETCH_HEAD
Already up to date.
✅ Repository updated with latest changes!


True

In [ ]:
def quick_commit(notebook_name, message=None):
    """Fixed commit function that ensures proper directory"""
    import datetime
    import shutil
    import os
    from google.colab import userdata
    import getpass
    import filecmp

    # Ensure we're in the right directory
    if not os.path.exists('/content/zindi_repo/.git'):
        print("Git repository not found. Run safe_setup_github() first.")
        return False

    os.chdir('/content/zindi_repo')
    print(f"Working in: {os.getcwd()}")

    if not message:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"Experiment update: {timestamp}"

    # Get GitHub token
    try:
        github_token = userdata.get('zindi_repo_git')
    except:
        github_token = getpass.getpass('Enter your GitHub token for push: ')

    # Notebook path
    notebook_path = f'/drive/My Drive/Colab Notebooks/Zindi - Huawei/{notebook_name}'

    # Check if notebook exists
    if not os.path.exists(notebook_path):
        print(f"Notebook not found at: {notebook_path}")
        return False

    try:
        # Ensure folder exists
        os.makedirs('huawei_networks', exist_ok=True)

        # Copy notebook
        destination_path = f'huawei_networks/{notebook_name}'
        shutil.copy(notebook_path, destination_path)
        print(f"Copied: {notebook_path} → {destination_path}")

        # Verify git status
        print("Git status before commit:")
        !git status --short

        # Set remote URL
        !git remote set-url origin https://{github_token}@github.com/fitzpk/zindi_repo.git

        # Commit and push
        !git add .
        !git commit -m "{message}"

        print("Pushing to GitHub...")
        !git push origin main

        print(f"Successfully committed '{notebook_name}': {message}")
        return True

    except Exception as e:
        print(f"Commit failed: {str(e)}")
        return False

quick_commit('huawei_baseline.ipynb', 'removed icons')

Working in: /content/zindi_repo
Copied: /drive/My Drive/Colab Notebooks/Zindi - Huawei/huawei_baseline.ipynb → huawei_networks/huawei_baseline.ipynb
Git status before commit:
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Pushing to GitHub...
Everything up-to-date
Successfully committed 'huawei_baseline.ipynb': removed icons


True

In [ ]:
# First, let's clean up the nested structure
import shutil
import os

# Remove the nested repo
if os.path.exists('/content/zindi_repo/zindi_repo'):
    shutil.rmtree('/content/zindi_repo/zindi_repo')
    print("Removed nested repository")

✅ Removed nested repository


In [ ]:
def debug_commit_issue(notebook_name):
    """Figure out why Git isn't detecting changes"""
    import os
    import filecmp
    import subprocess

    notebook_path = f'/drive/My Drive/Colab Notebooks/Zindi - Huawei/{notebook_name}'
    repo_path = f'/content/zindi_repo/huawei_networks/{notebook_name}'

    print("🔍 DEBUGGING GIT DETECTION ISSUE")
    print("=" * 50)

    # 1. Check file existence
    print(f"1. File check:")
    print(f"   Source exists: {os.path.exists(notebook_path)}")
    print(f"   Repo file exists: {os.path.exists(repo_path)}")

    if not os.path.exists(repo_path):
        print("   → Repo file doesn't exist - should be detected as NEW")
        return True

    # 2. Compare file sizes
    print(f"2. File sizes:")
    src_size = os.path.getsize(notebook_path)
    dst_size = os.path.getsize(repo_path)
    print(f"   Source: {src_size} bytes")
    print(f"   Repo:   {dst_size} bytes")
    print(f"   Different sizes: {src_size != dst_size}")

    # 3. Compare content
    print(f"3. Content comparison:")
    files_equal = filecmp.cmp(notebook_path, repo_path, shallow=False)
    print(f"   Files are identical: {files_equal}")

    # 4. Check file permissions
    print(f"4. File permissions:")
    src_stat = os.stat(notebook_path)
    dst_stat = os.stat(repo_path)
    print(f"   Source permissions: {oct(src_stat.st_mode)}")
    print(f"   Repo permissions:   {oct(dst_stat.st_mode)}")

    # 5. Check Git status specifically for this file
    print(f"5. Git status for this file:")
    os.chdir('/content/zindi_repo')
    !git status --short {repo_path}

    # 6. Force check with git diff
    print(f"6. Git diff (should show changes if any):")
    diff_result = subprocess.getoutput(f'git diff {repo_path}')
    if diff_result:
        print("   Changes detected by git diff:")
        print(f"   {diff_result[:200]}...")  # First 200 chars
    else:
        print("   No changes detected by git diff")

    # 7. Check if file is ignored by .gitignore
    print(f"7. Check if ignored:")
    check_ignore = subprocess.getoutput(f'git check-ignore {repo_path}')
    if check_ignore:
        print(f"   ❌ File is IGNORED by .gitignore: {check_ignore}")
    else:
        print("   ✅ File is not ignored")

    print("=" * 50)
    return not files_equal

# Run the debugger
debug_commit_issue('huawei_baseline.ipynb')

🔍 DEBUGGING GIT DETECTION ISSUE
1. File check:
   Source exists: True
   Repo file exists: True
2. File sizes:
   Source: 88939 bytes
   Repo:   76486 bytes
   Different sizes: True
3. Content comparison:
   Files are identical: False
4. File permissions:
   Source permissions: 0o100600
   Repo permissions:   0o100600
5. Git status for this file:
6. Git diff (should show changes if any):
   No changes detected by git diff
7. Check if ignored:
   ✅ File is not ignored


True

In [ ]:
def fix_git_tracking(notebook_name, message=None):
    """Completely reset Git's tracking of the file"""
    import os
    import datetime
    import shutil

    os.chdir('/content/zindi_repo')
    repo_path = f'huawei_networks/{notebook_name}'
    notebook_path = f'/drive/My Drive/Colab Notebooks/Zindi - Huawei/{notebook_name}'

    if not message:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"Experiment update: {timestamp}"

    print("🔄 Resetting Git tracking...")

    # 1. Remove the file from Git (but keep it locally)
    !git rm --cached {repo_path}

    # 2. Delete the file completely
    if os.path.exists(repo_path):
        os.remove(repo_path)

    # 3. Copy fresh from source
    shutil.copy2(notebook_path, repo_path)

    # 4. Add back to Git
    !git add {repo_path}

    # 5. Check if Git now sees the change
    status = !git status --porcelain
    print(f"📋 Git status after reset: {status}")

    if status:
        !git commit -m "{message}"
        !git push origin main
        print(f"✅ Successfully committed after reset: {message}")
        return True
    else:
        print("❌ Git STILL doesn't see changes - this is very weird")
        return False

# Run this fix
#fix_git_tracking('huawei_baseline.ipynb', 'Fixed Git tracking issue')